<div align="center">
  <h3><b>ESCUELA POLITÉCNICA NACIONAL</b></h3>
  <h3><b>FACULTAD DE INGENIERÍA EN SISTEMAS</b></h3>
  <h3><b>INGENIERÍA EN CIENCIAS DE LA COMPUTACIÓN</b></h3>
  <h3><b>RECUPERACIÓN DE LA INFORMACIÓN</b></h3>
</div>

---
**Nombre**   Mark Hernández        
**Fecha**    13/05/26  
**Docente**  Iván Carrera

# Ejercicio 10: Re-ranking

**Objetivo:** Implementar y evaluar un pipeline de Recuperación de Información en dos etapas, y analizar el impacto del re-ranking en la calidad del ranking.

## Parte 1. Preparación del corpus

* Cargar el corpus (documentos/pasajes).
* Cargar las consultas (queries).
* Cargar qrels (relevancia).

In [1]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader
import pandas as pd

c:\Users\mark_\Documents\ir26a\.venv\Lib\site-packages\beir\util.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [2]:
DATASET_NAME = "scifact"
DATA_DIR = "../data/beir_datasets"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{DATASET_NAME}.zip"
util.download_and_unzip(url, DATA_DIR)

../data/beir_datasets\scifact.zip: 100%|██████████| 2.69M/2.69M [00:07<00:00, 389kiB/s]


'../data/beir_datasets\\scifact'

In [3]:
dataset_path = DATA_DIR + "/" + DATASET_NAME
corpus, queries, qrels = GenericDataLoader(dataset_path).load(split="test")

100%|██████████| 5183/5183 [00:00<00:00, 60401.26it/s]


In [4]:
df_corpus = (
    pd.DataFrame.from_dict(corpus, orient="index")
      .reset_index()
      .rename(columns={"index": "doc_id"})
)

df_corpus

,doc_id,text,title
0,4983,Alterations of the architecture of cerebral wh...,Microstructural development of human newborn c...
1,5836,Myelodysplastic syndromes (MDS) are age-depend...,Induction of myelodysplasia by myeloid-derived...
2,7912,ID elements are short interspersed elements (S...,"BC1 RNA, the transcript from a master gene for..."
3,18670,DNA methylation plays an important role in bio...,The DNA Methylome of Human Peripheral Blood Mo...
4,19238,Two human Golli (for gene expressed in the oli...,The human myelin basic protein gene is include...
...,...,...,...
5178,195689316,BACKGROUND The main associations of body-mass ...,Body-mass index and cause-specific mortality i...
5179,195689757,A key aberrant biological difference between t...,Targeting metabolic remodeling in glioblastoma...
5180,196664003,A signaling pathway transmits information from...,Signaling architectures that transmit unidirec...
5181,198133135,AIMS Trabecular bone score (TBS) is a surrogat...,"Association between pre-diabetes, type 2 diabe..."


In [5]:
df_queries = (
    pd.DataFrame.from_dict(queries, orient="index", columns=["query"])
      .reset_index()
      .rename(columns={"index": "query_id"})
)

df_queries

,query_id,query
0,1,0-dimensional biomaterials show inductive prop...
1,3,"1,000 genomes project enables mapping of genet..."
2,5,1/2000 in UK have abnormal PrP positivity.
3,13,5% of perinatal mortality is due to low birth ...
4,36,A deficiency of vitamin B12 increases blood le...
...,...,...
295,1379,Women with a higher birth weight are more like...
296,1382,aPKCz causes tumour enhancement by affecting g...
297,1385,cSMAC formation enhances weak ligand signalling.
298,1389,mTORC2 regulates intracellular cysteine levels...


In [6]:
rows = []
for qid, docs in qrels.items():
    for doc_id, rel in docs.items():
        rows.append({
            "query_id": qid,
            "doc_id": doc_id,
            "relevance": rel
        })

df_qrels = pd.DataFrame(rows)
df_qrels

,query_id,doc_id,relevance
0,1,31715818,1
1,3,14717500,1
2,5,13734012,1
3,13,1606628,1
4,36,5152028,1
...,...,...,...
334,1379,17450673,1
335,1382,17755060,1
336,1385,306006,1
337,1389,23895668,1


In [7]:
# Elegimos una query cualquiera que tenga varios documentos relevantes
qid = "133"

print("Query:")
print(df_queries.loc[df_queries["query_id"] == qid, "query"].values[0])

print("\nDocumentos relevantes para esta query:")
df_qrels[(df_qrels["query_id"] == qid) & (df_qrels["relevance"] > 0)]

Query:
Assembly of invadopodia is triggered by focal generation of phosphatidylinositol-3,4-biphosphate and the activation of the nonreceptor tyrosine kinase Src.

Documentos relevantes para esta query:


,query_id,doc_id,relevance
31,133,38485364,1
32,133,6969753,1
33,133,17934082,1
34,133,16280642,1
35,133,12640810,1


## Parte 2. Retrieval inicial (baseline)

* Implementar retrieval inicial con BM25
* Obtener métricas: Recall@10 nDCG@10

In [9]:
from rank_bm25 import BM25Okapi
from beir.retrieval.evaluation import EvaluateRetrieval
from tqdm import tqdm

# 1. Preparamos el corpus para BM25 (Tokenización simple)
corpus_ids = list(corpus.keys())
tokenized_corpus = []

print("Tokenizando el corpus...")
for doc_id in tqdm(corpus_ids):
    # Unimos el título y el texto del documento para tener más contexto
    text = corpus[doc_id].get("title", "") + " " + corpus[doc_id].get("text", "")
    # Tokenización básica por espacios (se pasa todo a minúsculas)
    tokenized_corpus.append(text.lower().split())

# 2. Inicializamos el modelo BM25
print("Inicializando el índice BM25...")
bm25 = BM25Okapi(tokenized_corpus)

# 3. Obtenemos los scores para cada query
results = {}
print("Calculando scores para las queries...")
for query_id, query_text in tqdm(queries.items()):
    tokenized_query = query_text.lower().split()
    doc_scores = bm25.get_scores(tokenized_query)
    
    # BEIR espera un diccionario de diccionarios: {query_id: {doc_id: score}}
    results[query_id] = {corpus_ids[i]: float(score) for i, score in enumerate(doc_scores)}

Tokenizando el corpus...


  0%|          | 0/5183 [00:00<?, ?it/s]

100%|██████████| 5183/5183 [00:00<00:00, 20909.53it/s]


Inicializando el índice BM25...
Calculando scores para las queries...


100%|██████████| 300/300 [00:09<00:00, 30.80it/s]


Para obtener las métricas de Recall@k, Precision@k y nDGC@k usamos el evaluador de la librería Bier.

In [21]:
# 4. Evaluación de métricas
k_values = [10] # Nos interesa específicamente el top 10
ndcg, _map, recall, precision = EvaluateRetrieval.evaluate(qrels, results, k_values)

ndgc_char = 'NDCG@'+str(k_values[0])
recall_char = 'Recall@'+str(k_values[0])
precision_char = 'P@'+str(k_values[0])

print("\n--- Resultados Baseline (BM25) ---")
print(f"nDCG@{k_values}:  {ndcg[ndgc_char]:.4f}")
print(f"Precision@{k_values}: {precision[precision_char]:.4f}")
print(f"Recall@{k_values}: {recall[recall_char]:.4f}")


--- Resultados Baseline (BM25) ---
nDCG@[10]:  0.5597
Precision@[10]: 0.0763
Recall@[10]: 0.6862


A continuación se muestra el top 10 de los 100 mejores documentos recuperados con BM25 para una query en específico.

In [31]:
import pandas as pd
from IPython.display import display

# 1. VISUALIZACIÓN PREVIA: El Top 100 de BM25
qid_test = "1382" 
top_k_bm25 = 40

# Ordenamos el diccionario de resultados de BM25 y tomamos los 100 mejores
top_40_bm25_tuplas = sorted(results[qid_test].items(), key=lambda x: x[1], reverse=True)[:top_k_bm25]

# Creamos el DataFrame
df_bm25 = pd.DataFrame(top_40_bm25_tuplas, columns=["doc_id", "score_bm25"])
df_bm25.index = df_bm25.index + 1 # Para que el ranking empiece en 1 y no en 0
df_bm25.index.name = "rank_bm25"

print(f"--- Top 10 (de los 40 candidatos) recuperados por BM25 para la query {qid_test} ---")
display(df_bm25.head(10))

--- Top 10 (de los 40 candidatos) recuperados por BM25 para la query 1382 ---


,doc_id,score_bm25
rank_bm25,,
1,4138659,21.408055
2,1344498,20.480474
3,5256564,17.571587
4,3831884,16.944568
5,3085264,14.693063
6,6625693,13.681560
7,1554348,12.811321
8,28334217,12.304180
9,26688294,12.119411


## Parte 3. Implementación del re-ranking _cross-encoder_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [32]:
from sentence_transformers import CrossEncoder
import numpy as np

# 1. Cargamos el modelo Cross-Encoder
# Este modelo usa una arquitectura BERT pequeña entrenada con el dataset MS-MARCO
print("Cargando modelo Cross-Encoder...")
model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', max_length=512)

# 2. Configuramos el pipeline
top_k_bm25 = 40 # Solo re-rankearemos los mejores 40 documentos de la primera fase
reranked_results = {}

print(f"Re-rankeando el Top-{top_k_bm25} de documentos para cada query...")
for query_id, query_text in tqdm(queries.items()):
    # Obtenemos el Top K del diccionario "results" que calculamos con BM25
    top_docs_bm25 = sorted(results[query_id].items(), key=lambda item: item[1], reverse=True)[:top_k_bm25]
    
    # Extraemos solo los IDs de esos documentos
    doc_ids = [doc_id for doc_id, _ in top_docs_bm25]
    
    # Construimos los pares de oraciones (Query, Documento)
    sentence_pairs = []
    for doc_id in doc_ids:
        doc_text = corpus[doc_id].get("title", "") + " " + corpus[doc_id].get("text", "")
        sentence_pairs.append([query_text, doc_text])
        
    # El Cross-Encoder predice un nuevo score para cada par
    cross_scores = model.predict(sentence_pairs)
    
    # Guardamos los resultados actualizados para BEIR
    reranked_results[query_id] = {doc_ids[i]: float(cross_scores[i]) for i in range(len(doc_ids))}

Cargando modelo Cross-Encoder...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 7876.28it/s]


Re-rankeando el Top-40 de documentos para cada query...


 29%|██▉       | 87/300 [05:33<13:36,  3.83s/it]


KeyboardInterrupt: 

In [ ]:
# 3. VISUALIZACIÓN POSTERIOR: Comparación de Rankings
# Ordenamos los nuevos resultados del Cross-Encoder para nuestra query de prueba
top_100_cross_tuplas = sorted(reranked_results[qid_test].items(), key=lambda x: x[1], reverse=True)

# Creamos su DataFrame
df_cross = pd.DataFrame(top_100_cross_tuplas, columns=["doc_id", "score_cross"])
df_cross.index = df_cross.index + 1
df_cross.index.name = "rank_cross"

print(f"--- Top 10 (de los 100 candidatos) recuperados por Cross-Encoder para la query {qid_test} ---")
display(df_cross.head(10))

## Parte 4. Implementación del re-ranking _LTR_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

## Parte 5. Evaluación post re-ranking

Calcular métricas:
* nDCG@10
* MAP
* Recall@10